In [9]:
print("hola")

hola


# **1. Importación de Librerías**

In [10]:
from pathlib import Path

import hashlib
import json

import numpy as np
import pandas as pd

from PIL import Image
from IPython.display import display

# **2. Definir las rutas y los parámetros**

In [11]:
import os
from pathlib import Path
from google.colab import drive

# 1. Asegurar que Google Drive esté montado
drive.mount('/content/drive')

# 2. Definir la ruta raíz de tu MyDrive
drive_root = Path("/content/drive/MyDrive")

print("=" * 65)
print("📂 DIRECTORIO Y ESTRUCTURA DE CARPETAS EN TU GOOGLE DRIVE")
print("=" * 65)

if drive_root.exists():
    print(f"Ruta raíz: {drive_root}\n")

    # Recorrer de forma recursiva solo los directorios/carpetas
    for root, dirs, files in os.walk(drive_root):
        current_path = Path(root)

        # Omitir carpetas ocultas del sistema (que empiezan por '.')
        dirs[:] = [d for d in dirs if not d.startswith('.')]

        # Calcular profundidad para formatear visualmente como árbol
        depth = len(current_path.relative_to(drive_root).parts)
        indent = "   " * depth

        if depth == 0:
            print(f"📁 MyDrive/ ({current_path})")
        else:
            folder_name = current_path.name
            print(f"{indent}├── 📁 {folder_name}/")
            print(f"{indent}│   └─ Ruta completa: {current_path}")

    print("\n" + "=" * 65)
    print("✅ Exploración finalizada.")
else:
    print("❌ No se pudo encontrar el acceso a Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 DIRECTORIO Y ESTRUCTURA DE CARPETAS EN TU GOOGLE DRIVE
Ruta raíz: /content/drive/MyDrive

📁 MyDrive/ (/content/drive/MyDrive)
   ├── 📁 Colab Notebooks/
   │   └─ Ruta completa: /content/drive/MyDrive/Colab Notebooks
   ├── 📁 dataset-LZUPSD/
   │   └─ Ruta completa: /content/drive/MyDrive/dataset-LZUPSD
      ├── 📁 prepared/
      │   └─ Ruta completa: /content/drive/MyDrive/dataset-LZUPSD/prepared
      ├── 📁 prepared_1/
      │   └─ Ruta completa: /content/drive/MyDrive/dataset-LZUPSD/prepared_1
      ├── 📁 prepared_2/
      │   └─ Ruta completa: /content/drive/MyDrive/dataset-LZUPSD/prepared_2

✅ Exploración finalizada.


In [12]:
import sys
import zipfile
from pathlib import Path

# 1. DEFINICIÓN DE RUTAS Y CONFIGURACIÓN

# Ubicación base en tu Google Drive
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/dataset-LZUPSD")

# Archivo ZIP del dataset dentro de tu Drive
ZIP_PATH = DRIVE_BASE_DIR / "Seed dataset.zip"

# Ruta local temporal en Colab para extraer las imágenes
DATASET_DIR = Path("/content/Seed_dataset_extracted/Seed dataset")

# Carpeta de destino final en Drive para los datos procesados
OUTPUT_DIR = DRIVE_BASE_DIR / "prepared"

# Parámetros reproducibles
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# Extensiones de imagen admitidas
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".tif", ".tiff", ".webp"
}

# 2. LÓGICA DE VALIDACIÓN Y MANEJO DE ERRORES

def setup_environment():
    print("=" * 60)
    print("🚀 INICIANDO CONFIGURACIÓN Y VERIFICACIÓN DEL DATASET")
    print("=" * 60)

    # --- PASO A: Verificar / Extraer el Dataset de Entrada ---
    if DATASET_DIR.exists() and DATASET_DIR.is_dir():
        print(f"ℹ️ El dataset ya está listo en el entorno local: '{DATASET_DIR}'")
    else:
        print(f"🔍 Buscando el archivo del dataset en Drive: '{ZIP_PATH}'...")
        if ZIP_PATH.exists() and ZIP_PATH.is_file():
            try:
                print(f"📦 Descomprimiendo '{ZIP_PATH.name}' en '/content/Seed_dataset_extracted'...")
                extract_target = Path("/content/Seed_dataset_extracted")
                extract_target.mkdir(parents=True, exist_ok=True)

                with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
                    zip_ref.extractall(extract_target)

                print("✅ Descompresión completada con éxito.")
            except Exception as e:
                print(f"❌ ERROR CRÍTICO: Falló la descompresión del archivo ZIP: {e}")
                sys.exit(1)
        else:
            print(f"❌ ERROR CRÍTICO: No se encontró el dataset en '{ZIP_PATH}' ni en '{DATASET_DIR}'.")
            print("Verifica que 'Seed dataset.zip' esté dentro de la carpeta 'dataset-LZUPSD' en tu Drive.")
            sys.exit(1)

    # Confirmar existencia final del dataset
    if not (DATASET_DIR.exists() and DATASET_DIR.is_dir()):
        print(f"❌ ERROR CRÍTICO: La carpeta del dataset '{DATASET_DIR}' no es válida después del proceso.")
        sys.exit(1)

    # --- PASO B: Manejar la carpeta de salida (OUTPUT_DIR) ---
    final_output_dir = OUTPUT_DIR

    if final_output_dir.exists():
        counter = 1
        # Se genera un nuevo nombre tipo 'prepared_1', 'prepared_2', etc., si ya existe
        while final_output_dir.exists():
            final_output_dir = DRIVE_BASE_DIR / f"{OUTPUT_DIR.name}_{counter}"
            counter += 1
        print(f"⚠️ La carpeta de salida ya existía. Se creará una nueva carpeta en: '{final_output_dir}'")

    try:
        final_output_dir.mkdir(parents=True, exist_ok=False)
        print(f"✅ Carpeta de salida lista: '{final_output_dir}'")
    except Exception as e:
        print(f"❌ ERROR: No se pudo crear la carpeta de salida '{final_output_dir}': {e}")
        sys.exit(1)

    # --- PASO C: Resumen y estado final ---
    print("\n" + "=" * 60)
    print("ESTADO: CONFIGURACIÓN EXITOSA")
    print("=" * 60)
    print(f"• Dataset de origen : {DATASET_DIR}")
    print(f"• Directorio salida : {final_output_dir}")
    print(f"• Semilla (Random)  : {RANDOM_STATE}")
    print(f"• División (T/V/T)  : {TRAIN_SIZE:.0%} / {VAL_SIZE:.0%} / {TEST_SIZE:.0%}")
    print(f"• Extensiones       : {', '.join(sorted(IMAGE_EXTENSIONS))}")
    print("=" * 60)

    return DATASET_DIR, final_output_dir

# Ejecutar el script
DATASET_DIR, OUTPUT_DIR = setup_environment()

🚀 INICIANDO CONFIGURACIÓN Y VERIFICACIÓN DEL DATASET
ℹ️ El dataset ya está listo en el entorno local: '/content/Seed_dataset_extracted/Seed dataset'
⚠️ La carpeta de salida ya existía. Se creará una nueva carpeta en: '/content/drive/MyDrive/dataset-LZUPSD/prepared_3'
✅ Carpeta de salida lista: '/content/drive/MyDrive/dataset-LZUPSD/prepared_3'

ESTADO: CONFIGURACIÓN EXITOSA
• Dataset de origen : /content/Seed_dataset_extracted/Seed dataset
• Directorio salida : /content/drive/MyDrive/dataset-LZUPSD/prepared_3
• Semilla (Random)  : 42
• División (T/V/T)  : 70% / 15% / 15%
• Extensiones       : .bmp, .jpeg, .jpg, .png, .tif, .tiff, .webp


In [13]:
# 1. Contar total de imágenes en el dataset extraído
all_images = [
    f for f in DATASET_DIR.rglob("*")
    if f.suffix.lower() in IMAGE_EXTENSIONS
]
total_images = len(all_images)

# 2. Calcular la cantidad estimada para cada split
n_train = int(total_images * TRAIN_SIZE)
n_val = int(total_images * VAL_SIZE)
n_test = total_images - n_train - n_val  # Ajuste exacto para evitar sobrantes por redondeo

# 3. Mostrar el reporte en consola
print("\n" + "=" * 60)
print("📊 ESTIMACIÓN DE IMÁGENES POR SPLIT")
print("=" * 60)
print(f"📷 Total de imágenes encontradas : {total_images}")
print("-" * 60)
print(f"🔹 Train (Entrenamiento {TRAIN_SIZE:.0%}) : {n_train} imágenes")
print(f"🔸 Val   (Validación    {VAL_SIZE:.0%}) : {n_val} imágenes")
print(f"🔹 Test  (Prueba        {TEST_SIZE:.0%}) : {n_test} imágenes")
print("=" * 60)


📊 ESTIMACIÓN DE IMÁGENES POR SPLIT
📷 Total de imágenes encontradas : 4496
------------------------------------------------------------
🔹 Train (Entrenamiento 70%) : 3147 imágenes
🔸 Val   (Validación    15%) : 674 imágenes
🔹 Test  (Prueba        15%) : 675 imágenes


In [14]:
import numpy as np

assert DATASET_DIR.exists(), (
    f"No existe el directorio: {DATASET_DIR}"
)

assert DATASET_DIR.is_dir(), (
    f"La ruta no es un directorio: {DATASET_DIR}"
)

assert np.isclose(
    TRAIN_SIZE + VAL_SIZE + TEST_SIZE,
    1.0
), "Los porcentajes deben sumar 1."

assert RANDOM_STATE == 42

print("Configuración validada correctamente.")

Configuración validada correctamente.


# **2. Reconstruir el inventario de imágenes**

In [15]:
import pandas as pd

# Obtener las categorías a partir de las carpetas
classes = sorted([
    folder.name
    for folder in DATASET_DIR.iterdir()
    if folder.is_dir()
])

# Asignar un identificador reproducible a cada categoría
class_to_id = {
    class_name: class_id
    for class_id, class_name in enumerate(classes)
}

# Construir el inventario
records = []

for class_name in classes:

    class_dir = DATASET_DIR / class_name
    class_id = class_to_id[class_name]

    for file_path in sorted(class_dir.iterdir()):

        if (
            file_path.is_file()
            and file_path.suffix.lower() in IMAGE_EXTENSIONS
        ):
            records.append({
                "image_path": str(file_path),
                "class_name": class_name,
                "class_id": class_id
            })

df_images = pd.DataFrame(records)

print("Categorías encontradas:", len(classes))
print("Imágenes registradas:", len(df_images))

display(df_images.head())

Categorías encontradas: 88
Imágenes registradas: 4496


,image_path,class_name,class_id
0,/content/Seed_dataset_extracted/Seed dataset/A...,Achnatherum inebrians,0
1,/content/Seed_dataset_extracted/Seed dataset/A...,Achnatherum inebrians,0
2,/content/Seed_dataset_extracted/Seed dataset/A...,Achnatherum inebrians,0
3,/content/Seed_dataset_extracted/Seed dataset/A...,Achnatherum inebrians,0
4,/content/Seed_dataset_extracted/Seed dataset/A...,Achnatherum inebrians,0


In [16]:
# Ver cuántas imágenes hay por cada una de las 88 clases
print("Distribución de imágenes por clase:")
display(df_images['class_name'].value_counts())

Distribución de imágenes por clase:


,count
class_name,
Medicago sativa,106
Lespedeza daurica,100
Puccinellia distans,94
Trifolium repens,92
Agriophyllum squarrosum,89
...,...
Agropyron elongatum,37
Astragalus laxmannii,37
Caragana korshinskii,36


In [17]:
assert len(classes) == 88, (
    f"Se esperaban 88 categorías, pero se encontraron {len(classes)}."
)

assert len(df_images) == 4496, (
    f"Se esperaban 4496 imágenes, pero se encontraron {len(df_images)}."
)

assert df_images["image_path"].is_unique, (
    "Se encontraron rutas de imagen repetidas."
)

assert not df_images.isnull().any().any(), (
    "El inventario contiene valores nulos."
)

assert set(df_images["class_name"]) == set(classes)

print("Inventario reconstruido y validado correctamente.")

Inventario reconstruido y validado correctamente.


# **3. Detectar y Tratar Duplicados**

In [18]:
import hashlib

# Copia de trabajo: conservamos df_images intacto
df_inventory = df_images.copy()

def calculate_md5(image_path):
    """Calcula el hash MD5 de un archivo."""
    hash_md5 = hashlib.md5()

    with open(image_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            hash_md5.update(chunk)

    return hash_md5.hexdigest()


df_inventory["file_hash"] = df_inventory["image_path"].apply(
    calculate_md5
)

print("Hashes calculados:", len(df_inventory))
print("Hashes únicos:", df_inventory["file_hash"].nunique())

Hashes calculados: 4496
Hashes únicos: 4494


## **3.1 Identificar los grupos duplicados**

In [19]:
duplicate_groups = (
    df_inventory
    .groupby("file_hash")
    .filter(lambda group: len(group) > 1)
    .sort_values(["file_hash", "image_path"])
)

display(
    duplicate_groups[
        ["image_path", "class_name", "file_hash"]
    ]
)

print(
    "Imágenes involucradas:",
    len(duplicate_groups)
)

print(
    "Grupos de duplicados:",
    duplicate_groups["file_hash"].nunique()
)

,image_path,class_name,file_hash
693,/content/Seed_dataset_extracted/Seed dataset/A...,Artemisia sphaerocephala,9b8dda01c44b3185f8775aabc6b4a48c
696,/content/Seed_dataset_extracted/Seed dataset/A...,Artemisia sphaerocephala,9b8dda01c44b3185f8775aabc6b4a48c
644,/content/Seed_dataset_extracted/Seed dataset/A...,Artemisia sphaerocephala,f945197d632d9145b7f285b4cd0336ed
4313,/content/Seed_dataset_extracted/Seed dataset/T...,Trifolium repens,f945197d632d9145b7f285b4cd0336ed


Imágenes involucradas: 4
Grupos de duplicados: 2


## **3.2 Definir el tratamiento**

In [20]:
"""
Si las imágenes idénticas tienen etiquetas diferentes, se excluyen todas las copias del grupo.

Si tienen la misma etiqueta, se conserva una copia y se excluyen las restantes.

Las imágenes no duplicadas se conservan.
"""

excluded_records = []

for file_hash, group in duplicate_groups.groupby("file_hash"):

    group = group.sort_values("image_path")
    labels = group["class_name"].unique()

    if len(labels) > 1:
        # Conflicto de etiquetas: excluir todas las copias
        for _, row in group.iterrows():
            record = row.to_dict()
            record["exclusion_reason"] = (
                "Imagen idéntica asociada a etiquetas diferentes"
            )
            excluded_records.append(record)

    else:
        # Misma etiqueta: conservar la primera copia
        # y excluir las demás
        for _, row in group.iloc[1:].iterrows():
            record = row.to_dict()
            record["exclusion_reason"] = (
                "Duplicado exacto dentro de la misma categoría"
            )
            excluded_records.append(record)


df_excluded = pd.DataFrame(excluded_records)

excluded_paths = set(df_excluded["image_path"])

df_clean = (
    df_inventory[
        ~df_inventory["image_path"].isin(excluded_paths)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Imágenes originales:", len(df_inventory))
print("Imágenes excluidas:", len(df_excluded))
print("Imágenes restantes:", len(df_clean))
print("Categorías restantes:", df_clean["class_name"].nunique())

Imágenes originales: 4496
Imágenes excluidas: 3
Imágenes restantes: 4493
Categorías restantes: 88


## **3.4 Revisar Exlusiones**

In [21]:
display(
    df_excluded[
        ["image_path", "class_name", "file_hash", "exclusion_reason"]
    ]
)

,image_path,class_name,file_hash,exclusion_reason
0,/content/Seed_dataset_extracted/Seed dataset/A...,Artemisia sphaerocephala,9b8dda01c44b3185f8775aabc6b4a48c,Duplicado exacto dentro de la misma categoría
1,/content/Seed_dataset_extracted/Seed dataset/A...,Artemisia sphaerocephala,f945197d632d9145b7f285b4cd0336ed,Imagen idéntica asociada a etiquetas diferentes
2,/content/Seed_dataset_extracted/Seed dataset/T...,Trifolium repens,f945197d632d9145b7f285b4cd0336ed,Imagen idéntica asociada a etiquetas diferentes


# **4. Validar el dataset definitivo**

In [22]:
# no queden hashes duplicados y que se conserven las 88 categorías.
assert df_clean["file_hash"].is_unique, (
    "Todavía existen imágenes duplicadas."
)

assert df_clean["image_path"].is_unique, (
    "Hay rutas repetidas."
)

assert df_clean["class_name"].nunique() == 88, (
    "El número de categorías cambió."
)

assert df_clean["class_id"].nunique() == 88

assert not df_clean[
    ["image_path", "class_name", "class_id", "file_hash"]
].isnull().any().any()

print("Dataset definitivo validado.")
print("Imágenes:", len(df_clean))
print("Categorías:", df_clean["class_name"].nunique())
print("Hashes duplicados:", df_clean["file_hash"].duplicated().sum())

Dataset definitivo validado.
Imágenes: 4493
Categorías: 88
Hashes duplicados: 0


In [23]:
class_counts_clean = (
    df_clean
    .groupby(["class_id", "class_name"])
    .size()
    .reset_index(name="image_count")
    .sort_values("class_name")
)

display(class_counts_clean)

print("Mínimo por categoría:", class_counts_clean["image_count"].min())
print("Máximo por categoría:", class_counts_clean["image_count"].max())

,class_id,class_name,image_count
0,0,Achnatherum inebrians,42
1,1,Achnatherum splendens,43
2,2,Agriophyllum squarrosum,89
3,3,Agropyron cristatum,56
4,4,Agropyron elongatum,37
...,...,...,...
83,83,Trifolium repens,91
84,84,Triticale,48
85,85,Vicia sativa,46
86,86,Vicia villosa,39


Mínimo por categoría: 35
Máximo por categoría: 106


# **5. Realizar el Split**

In [24]:
from sklearn.model_selection import train_test_split

# Primero separamos el conjunto de prueba
df_train_val, df_test = train_test_split(
    df_clean,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_clean["class_id"]
)

# Proporción de validación respecto al conjunto restante
val_fraction = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)

# Separamos entrenamiento y validación
df_train, df_val = train_test_split(
    df_train_val,
    test_size=val_fraction,
    random_state=RANDOM_STATE,
    stratify=df_train_val["class_id"]
)

# Reiniciar índices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print("Train:", len(df_train))
print("Validation:", len(df_val))
print("Test:", len(df_test))
print("Total:", len(df_train) + len(df_val) + len(df_test))

Train: 3145
Validation: 674
Test: 674
Total: 4493


In [25]:
total = len(df_train) + len(df_val) + len(df_test)

print(f"Train:      {len(df_train):4d} ({len(df_train) / total:.2%})")
print(f"Validation: {len(df_val):4d} ({len(df_val) / total:.2%})")
print(f"Test:       {len(df_test):4d} ({len(df_test) / total:.2%})")

assert len(df_train) + len(df_val) + len(df_test) == len(df_clean)

for name, split_df in [
    ("train", df_train),
    ("validation", df_val),
    ("test", df_test)
]:
    assert split_df["class_id"].nunique() == 88, (
        f"Hay categorías ausentes en {name}."
    )

print("\nLas tres particiones contienen las 88 categorías.")

Train:      3145 (70.00%)
Validation:  674 (15.00%)
Test:        674 (15.00%)

Las tres particiones contienen las 88 categorías.


In [26]:
train_hashes = set(df_train["file_hash"])
val_hashes = set(df_val["file_hash"])
test_hashes = set(df_test["file_hash"])

assert train_hashes.isdisjoint(val_hashes)
assert train_hashes.isdisjoint(test_hashes)
assert val_hashes.isdisjoint(test_hashes)

print("No existen hashes compartidos entre las particiones.")

No existen hashes compartidos entre las particiones.


# **6. Guardar los archivos en Google Drive**

In [27]:
def prepare_split_for_export(df):
    """Prepara las columnas que se guardarán en CSV."""
    result = df[
        ["image_path", "class_name", "class_id"]
    ].copy()

    result["image_path"] = result["image_path"].apply(
        lambda path: Path(path).relative_to(DATASET_DIR).as_posix()
    )

    return result


train_export = prepare_split_for_export(df_train)
val_export = prepare_split_for_export(df_val)
test_export = prepare_split_for_export(df_test)

excluded_export = df_excluded.copy()

if not excluded_export.empty:
    excluded_export["image_path"] = excluded_export["image_path"].apply(
        lambda path: Path(path).relative_to(DATASET_DIR).as_posix()
    )

In [28]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_export.to_csv(
    OUTPUT_DIR / "train.csv",
    index=False
)

val_export.to_csv(
    OUTPUT_DIR / "val.csv",
    index=False
)

test_export.to_csv(
    OUTPUT_DIR / "test.csv",
    index=False
)

excluded_export.to_csv(
    OUTPUT_DIR / "excluded_images.csv",
    index=False
)

print("Archivos CSV guardados en:", OUTPUT_DIR)

Archivos CSV guardados en: /content/drive/MyDrive/dataset-LZUPSD/prepared_3


# **7. Guardar y verificar configuraciones**

In [29]:
import json

config = {
    "dataset_name": "LZUPSD",
    "random_state": RANDOM_STATE,
    "train_size": TRAIN_SIZE,
    "validation_size": VAL_SIZE,
    "test_size": TEST_SIZE,
    "num_classes": int(df_clean["class_name"].nunique()),
    "num_images_original": int(len(df_inventory)),
    "num_images_excluded": int(len(df_excluded)),
    "num_images_final": int(len(df_clean)),
    "image_extensions": sorted(IMAGE_EXTENSIONS),
    "class_to_id": {
        str(class_name): int(class_id)
        for class_name, class_id in class_to_id.items()
    }
}

with open(OUTPUT_DIR / "config.json", "w", encoding="utf-8") as file:
    json.dump(config, file, indent=4, ensure_ascii=False)

print("Configuración guardada.")

Configuración guardada.


In [30]:
print("Contenido del directorio de salida:\n")

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size / 1024:.2f} KB")

Contenido del directorio de salida:

config.json: 3.35 KB
excluded_images.csv: 0.48 KB
test.csv: 37.59 KB
train.csv: 175.09 KB
val.csv: 37.57 KB


In [34]:
df_train_check = pd.read_csv(OUTPUT_DIR / "train.csv")
df_val_check = pd.read_csv(OUTPUT_DIR / "val.csv")
df_test_check = pd.read_csv(OUTPUT_DIR / "test.csv")

df_train_check["image_path"] = df_train_check["image_path"].apply(
    lambda path: str(DATASET_DIR / path)
)

df_val_check["image_path"] = df_val_check["image_path"].apply(
    lambda path: str(DATASET_DIR / path)
)

df_test_check["image_path"] = df_test_check["image_path"].apply(
    lambda path: str(DATASET_DIR / path)
)

In [35]:
train_paths = set(df_train_check["image_path"])
val_paths = set(df_val_check["image_path"])
test_paths = set(df_test_check["image_path"])

print("Solapamientos entre particiones")
print("-" * 40)

print("Train - Val:", len(train_paths & val_paths))
print("Train - Test:", len(train_paths & test_paths))
print("Val - Test:", len(val_paths & test_paths))

Solapamientos entre particiones
----------------------------------------
Train - Val: 0
Train - Test: 0
Val - Test: 0
